# 2025-06-19. Mulvirisk 2 - Validation

In [31]:
from rdflib import Graph
import pandas as pd

In [2]:
sites = Graph()
sites.parse(open("../results/2025-06-17.mulvirisk-model2/sites.ttl"))

libraries = Graph()
libraries.parse(open("../results/2025-06-17.mulvirisk-model2/libraries.ttl"))

hosts = Graph()
hosts.parse(open("../results/2025-06-17.mulvirisk-model2/host.ttl"))

virus = Graph()
virus.parse(open("../results/2025-06-17.mulvirisk-model2/virus.ttl"))

bacteria = Graph()
bacteria.parse(open("../results/2025-06-17.mulvirisk-model2/bacteria.ttl"))

<Graph identifier=N92bd816608a84d47b8347bf5abac4dc9 (<class 'rdflib.graph.Graph'>)>

In [3]:
G = sites + libraries + hosts + virus + bacteria

In [4]:
res = G.query(
    """
    PREFIX mvrsite: <http://localhost:8000/site/>
    PREFIX wlo: <http://purl.org/ontology/wo/>
    SELECT ?site
    WHERE {
        ?site wlo:habitat "Crop"
    }

    """
)
for row in res:
    print("%s" % row)

http://localhost:8000/site/H1
http://localhost:8000/site/C2
http://localhost:8000/site/Z2
http://localhost:8000/site/M1
http://localhost:8000/site/M4
http://localhost:8000/site/H2
http://localhost:8000/site/M3
http://localhost:8000/site/M2
http://localhost:8000/site/C1
http://localhost:8000/site/Z1
http://localhost:8000/site/H3


In [5]:
res = G.query(
    """
    PREFIX mvrtaxon: <http://localhost:8000/taxon/>
    PREFIX mvrlib: <http://localhost:8000/lib/>
    PREFIX mvront: <http://localhost:8000/ont/>
    PREFIX wlo: <http://purl.org/ontology/wo/>
    PREFIX emi: <https://purl.org/emi#>
    
    SELECT ?organism ?lib
    WHERE {
        ?organism emi:inTaxon mvrtaxon:000000000195 .
        ?lib mvront:Reported ?organism 
    }

    """
)
for row in res:
    print("%s %s" % row)

http://localhost:8000/hit/000000000001 http://localhost:8000/library/PV059
http://localhost:8000/hit/000000000003 http://localhost:8000/library/PV182
http://localhost:8000/hit/000000000002 http://localhost:8000/library/PV159
http://localhost:8000/hit/000000000004 http://localhost:8000/library/PV498


## Test 1 - Are all libraries reporting organisms defined as libraries?

In [6]:
res1 = G.query(
    """
    PREFIX mvrtaxon: <http://localhost:8000/taxon/>
    PREFIX mvrlib: <http://localhost:8000/lib/>
    PREFIX mvront: <http://localhost:8000/ont/>
    PREFIX wlo: <http://purl.org/ontology/wo/>
    PREFIX emi: <https://purl.org/emi#>
    
    SELECT DISTINCT ?lib
    WHERE {
        ?lib a mvront:Library
    }

    """
)
res1 = ["%s" % r for r in res1]

In [7]:
res2 = G.query(
    """
    PREFIX mvrtaxon: <http://localhost:8000/taxon/>
    PREFIX mvrlib: <http://localhost:8000/library/>
    PREFIX mvront: <http://localhost:8000/ont/>
    PREFIX wlo: <http://purl.org/ontology/wo/>
    PREFIX emi: <https://purl.org/emi#>
    
    SELECT DISTINCT ?lib
    WHERE {
        ?lib mvront:Reported ?organism
    }

    """
)
res2 = ["%s" % r for r in res2]

In [8]:
len(res1) == len(res2)

False

In [9]:
len(res1)

323

In [10]:
len(res2)

314

In [11]:
for item in res1:
    if item not in res2:
        print(item)

http://localhost:8000/library/PV569
http://localhost:8000/library/PV564
http://localhost:8000/library/PV581
http://localhost:8000/library/PV571
http://localhost:8000/library/PV579
http://localhost:8000/library/PV563
http://localhost:8000/library/PV582
http://localhost:8000/library/PV580
http://localhost:8000/library/PV574
http://localhost:8000/library/PV454


In [12]:
for item in res2:
    if item not in res1:
        print(item)

http://localhost:8000/library/PV067


Intringuing. There are some libraries that were defined but did not report any connections, and libraries that reported connections but they were not defined. 

In [13]:
list(G.query(
    """
    PREFIX mvrtaxon: <http://localhost:8000/taxon/>
    PREFIX mvrlib: <http://localhost:8000/library/>
    PREFIX mvront: <http://localhost:8000/ont/>
    PREFIX wlo: <http://purl.org/ontology/wo/>
    PREFIX emi: <https://purl.org/emi#>
    
    SELECT ?s ?p ?o
    WHERE {
        mvrlib:PV566 ?p ?o
    }
    """
))

[(None,
  rdflib.term.URIRef('http://purl.org/dc/elements/1.1/date'),
  rdflib.term.Literal('19/11/15')),
 (None,
  rdflib.term.URIRef('http://purl.org/dc/elements/1.1/date'),
  rdflib.term.Literal('10/11/16')),
 (None,
  rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
  rdflib.term.URIRef('http://localhost:8000/ont/Library')),
 (None,
  rdflib.term.URIRef('http://localhost:8000/ont/Obtained_from'),
  rdflib.term.URIRef('http://localhost:8000/host/000000000108')),
 (None,
  rdflib.term.URIRef('http://localhost:8000/ont/nextracts'),
  rdflib.term.Literal('7')),
 (None,
  rdflib.term.URIRef('http://localhost:8000/ont/Sampled_from'),
  rdflib.term.URIRef('http://localhost:8000/site/E2')),
 (None,
  rdflib.term.URIRef('http://localhost:8000/ont/Reported'),
  rdflib.term.URIRef('http://localhost:8000/hit/000000003057'))]

## Test 2. Obtaining a list of Host-hits with their types

In [14]:
res = G.query(
    """
    PREFIX mvrtaxon: <http://localhost:8000/taxon/>
    PREFIX mvrlib: <http://localhost:8000/library/>
    PREFIX mvront: <http://localhost:8000/ont/>
    PREFIX wlo: <http://purl.org/ontology/wo/>
    PREFIX emi: <https://purl.org/emi#>
    
    SELECT ?hit ?host ?hittype
    WHERE {
        
        ?lib a mvront:Library .
        ?lib mvront:Reported ?organism .
        ?organism emi:inTaxon ?hititem .
        ?hititem emi:scientificName ?hit .
        ?hititem mvront:OrganismKingdom ?hittype .
        ?lib mvront:Obtained_from ?hostorg .
        ?hostorg emi:inTaxon ?hostitem .
        ?hostitem emi:scientificName ?host .
    }
    """
)
# 
#  .
# ?lib a mvront:Library .
        # ?host a mvront:Host .
#  .
        #?hosthit emi:inTaxon ?host .
for item in list(res):
    print("%s, %s, %s " % item)

Tobacco mosaic virus, Papaver rhoeas, Virus 
Cucumber mosaic virus RNA 3, Papaver rhoeas, Virus 
Tobacco mild green mosaic virus, Papaver rhoeas, Virus 
Eggplant mottled dwarf virus, Papaver rhoeas, Virus 
Clavibacter michiganensis, Papaver rhoeas, Bacteria 
Aureimonas sp. AU12, Papaver rhoeas, Bacteria 
Methylobacterium sp. Leaf125, Papaver rhoeas, Bacteria 
Rhizobiales bacterium 63-22, Papaver rhoeas, Bacteria 
Curtobacterium sp. MMLR14_010, Papaver rhoeas, Bacteria 
Cnuibacter physcomitrellae, Papaver rhoeas, Bacteria 
uncultured Clostridium sp., Papaver rhoeas, Bacteria 
Frigoribacterium sp. Leaf164, Papaver rhoeas, Bacteria 
Watermelon mosaic virus, Jasminum fruticans, Virus 
Cucumber mosaic virus RNA 3, Jasminum fruticans, Virus 
uncultured Clostridium sp., Jasminum fruticans, Bacteria 
Psychrobacter alimentarius, Klasea pinnatifida, Bacteria 
uncultured Clostridium sp., Klasea pinnatifida, Bacteria 
Frigoribacterium sp. Leaf164, Klasea pinnatifida, Bacteria 
Sphingomonas sp. Lea

In [15]:
res = G.query(
    """
    PREFIX mvrtaxon: <http://localhost:8000/taxon/>
    PREFIX mvrlib: <http://localhost:8000/library/>
    PREFIX mvront: <http://localhost:8000/ont/>
    PREFIX wlo: <http://purl.org/ontology/wo/>
    PREFIX emi: <https://purl.org/emi#>
    
    SELECT ?hit
    WHERE {
        ?hit a emi:Organism .
        ?hit emi:inTaxon ?taxon .
        ?taxon mvront:OrganismKingdom "Virus"
        
    }
    """
)
# 
#  .
# ?lib a mvront:Library .
        # ?host a mvront:Host .
#  .
        #?hosthit emi:inTaxon ?host .
for item in list(res):
    print("%s" % item)

http://localhost:8000/hit/000000000222
http://localhost:8000/hit/000000000218
http://localhost:8000/hit/000000000220
http://localhost:8000/hit/000000000221
http://localhost:8000/hit/000000000219
http://localhost:8000/hit/000000000236
http://localhost:8000/hit/000000000237
http://localhost:8000/hit/000000001351
http://localhost:8000/hit/000000001350
http://localhost:8000/hit/000000000109
http://localhost:8000/hit/000000001040
http://localhost:8000/hit/000000001039
http://localhost:8000/hit/000000000489
http://localhost:8000/hit/000000001056
http://localhost:8000/hit/000000001052
http://localhost:8000/hit/000000001050
http://localhost:8000/hit/000000001058
http://localhost:8000/hit/000000001057
http://localhost:8000/hit/000000001054
http://localhost:8000/hit/000000001055
http://localhost:8000/hit/000000001053
http://localhost:8000/hit/000000001051
http://localhost:8000/hit/000000000484
http://localhost:8000/hit/000000001385
http://localhost:8000/hit/000000000614
http://localhost:8000/hit

In [16]:
res = G.query(
    """
    PREFIX mvrtaxon: <http://localhost:8000/taxon/>
    PREFIX mvrlib: <http://localhost:8000/library/>
    PREFIX mvront: <http://localhost:8000/ont/>
    PREFIX wlo: <http://purl.org/ontology/wo/>
    PREFIX emi: <https://purl.org/emi#>
    
    SELECT ?s ?p
    WHERE {
        mvrlib:PV094 a mvront:Library .
        ?s ?p mvrlib:PV094
    
    }
    """
)
# ?lib mvront:Obtained_from ?host .
# ?lib mvront:Reported ?hit .
# ?lib a mvront:Library .
        # ?host a mvront:Host .
#  .
        #?hosthit emi:inTaxon ?host .
for item in list(res):
    print("%s %s" % (item))

In [21]:
res = G.query(
    """
    PREFIX mvrtaxon: <http://localhost:8000/taxon/>
    PREFIX mvrlib: <http://localhost:8000/library/>
    PREFIX mvront: <http://localhost:8000/ont/>
    PREFIX wlo: <http://purl.org/ontology/wo/>
    PREFIX emi: <https://purl.org/emi#>
    
    SELECT (COUNT(?lib) AS ?count) ?site
    WHERE {
        ?site a mvront:Site .
        ?lib mvront:Sampled_from ?site .
    }   GROUP BY ?site 
    """
)
for item in list(res):
    print("%s\t%s" % item)

6	http://localhost:8000/site/C2
18	http://localhost:8000/site/E2
12	http://localhost:8000/site/Q3
4	http://localhost:8000/site/M2
25	http://localhost:8000/site/Q2
13	http://localhost:8000/site/Q4
14	http://localhost:8000/site/E1
6	http://localhost:8000/site/Z1
5	http://localhost:8000/site/H3
40	http://localhost:8000/site/L1
4	http://localhost:8000/site/Z2
24	http://localhost:8000/site/Q1
35	http://localhost:8000/site/L2
23	http://localhost:8000/site/L4
4	http://localhost:8000/site/M4
35	http://localhost:8000/site/L3
8	http://localhost:8000/site/M1
4	http://localhost:8000/site/H2
5	http://localhost:8000/site/C1
13	http://localhost:8000/site/E3
4	http://localhost:8000/site/H1
5	http://localhost:8000/site/M3
19	http://localhost:8000/site/E4


In [36]:
res = G.query(
    """
    PREFIX mvrtaxon: <http://localhost:8000/taxon/>
    PREFIX mvrlib: <http://localhost:8000/library/>
    PREFIX mvront: <http://localhost:8000/ont/>
    PREFIX wlo: <http://purl.org/ontology/wo/>
    PREFIX emi: <https://purl.org/emi#>
    
    SELECT ?site ?taxid (COUNT(?lib) AS ?nhits) 
    WHERE {
        ?site a mvront:Site .
        ?lib mvront:Sampled_from ?site .
        ?lib mvront:Reported ?organism .
        ?organism emi:inTaxon ?hit .
        ?hit mvront:TaxonID ?taxid
    } GROUP BY ?site ?taxid
    """
)
pd.DataFrame(res, columns=list(res.vars))

,site,taxid,nhits
0,http://localhost:8000/site/C2,12242,2
1,http://localhost:8000/site/C2,12241,1
2,http://localhost:8000/site/C2,196914,1
3,http://localhost:8000/site/C2,1735685,1
4,http://localhost:8000/site/C2,147709,2
...,...,...,...
1455,http://localhost:8000/site/E4,192203,1
1456,http://localhost:8000/site/E4,1809767,1
1457,http://localhost:8000/site/E4,909827,1
1458,http://localhost:8000/site/E4,1817526,1
